In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers datasets edge-tts nest_asyncio torchaudio jiwer -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 22.9 MB/s eta 0:00:00


In [ ]:
import os
import ast
import torch
import torchaudio
import pandas as pd
import numpy as np
from difflib import SequenceMatcher
from transformers import AutoProcessor, AutoModelForCTC
import IPython.display as ipd

print("✅ All imports done")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ No GPU - switch runtime!'}")

✅ All imports done
✅ GPU: ❌ No GPU - switch runtime!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/URTOX_v2.csv")

print(f"✅ Dataset loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset loaded: (14337, 8)
Columns: ['id', 'text', 'label', 'sub_label', 'toxic_spans', 'tokens', 'toxic_list', 'BIO_tags']

Label distribution:
label
toxic        7751
non_toxic    6586
Name: count, dtype: int64


In [ ]:
def safe_parse(x):
    try:
        return ast.literal_eval(str(x))
    except:
        return []

df['tokens']      = df['tokens'].apply(safe_parse)
df['BIO_tags']    = df['BIO_tags'].apply(safe_parse)
df['toxic_spans'] = df['toxic_spans'].apply(safe_parse)
df['toxic_list']  = df['toxic_list'].apply(safe_parse)

print("✅ Columns parsed successfully")
print(f"Sample tokens: {df['tokens'].iloc[4]}")
print(f"Sample BIO:    {df['BIO_tags'].iloc[4]}")

✅ Columns parsed successfully
Sample tokens: ['بےادب', 'ہرگز', 'عالم', 'نہیں', 'ہوسکتا']
Sample BIO:    ['B-Toxic', 'O', 'O', 'O', 'O']


In [ ]:
AUDIO_DIR = "/content/drive/MyDrive/urdu_toxic_audio _og"

df['audio_path'] = df['id'].apply(
    lambda x: f"{AUDIO_DIR}/{x}.mp3"
    if os.path.exists(f"{AUDIO_DIR}/{x}.mp3") else None
)

found    = df['audio_path'].notna().sum()
missing  = df['audio_path'].isna().sum()

print(f"✅ Audio files found:   {found}/{len(df)}")
print(f"⚠️  Missing audio files: {missing}")

# Quick sanity check - show a few paths
print(f"\nSample paths:")
print(df[df['audio_path'].notna()]['audio_path'].head(3).tolist())

✅ Audio files found:   14337/14337
⚠️  Missing audio files: 0

Sample paths:
['/content/drive/MyDrive/urdu_toxic_audio _og/5003.mp3', '/content/drive/MyDrive/urdu_toxic_audio _og/5005.mp3', '/content/drive/MyDrive/urdu_toxic_audio _og/5010.mp3']


In [ ]:
import os

folder_path = "/content/drive/MyDrive/urdu_toxic_audio _og"

total_files = len([
    f for f in os.listdir(folder_path)
    if os.path.isfile(os.path.join(folder_path, f))
])

print("Total files:", total_files)

Total files: 14337


In [ ]:
sample = df[df['audio_path'].notna()].iloc[0]

print(f"Text:  {sample['text']}")
print(f"Label: {sample['label']}")
print(f"BIO:   {sample['BIO_tags']}")
print(f"Path:  {sample['audio_path']}")

ipd.display(ipd.Audio(sample['audio_path']))

Text:  ایک جملہ ھی حضرت علی علیہ السلام کی فضیلت کے لئے کافی ہے۔ " من کنت مولا فھذا علی مولا "  اپنے مولا سے کوئی کیسے افضل ھو سکتا ھے؟
Label: non_toxic
BIO:   ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Toxic', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Path:  /content/drive/MyDrive/urdu_toxic_audio _og/5003.mp3


In [ ]:
audio_df = df[df['audio_path'].notna()].copy()

audio_df['transcription']   = audio_df['text']
audio_df['new_tokens']      = audio_df['tokens']
audio_df['new_BIO_tags']    = audio_df['BIO_tags']
audio_df['toxic_preserved'] = True

total      = len(audio_df)
toxic_rows = audio_df[audio_df['new_BIO_tags'].apply(lambda x: 'B-Toxic' in x)]

print(f"✅ Done instantly!")
print(f"Total rows:           {total}")
print(f"Rows with toxic tags: {len(toxic_rows)}")
print(f"Label distribution:\n{audio_df['label'].value_counts()}")

✅ Done instantly!
Total rows:           14337
Rows with toxic tags: 6298
Label distribution:
label
toxic        7751
non_toxic    6586
Name: count, dtype: int64


In [ ]:
audio_df = pd.DataFrame(audio_df)  # already a df, this is fine

save_path = "/content/drive/MyDrive/urdu_toxic_audio_dataset.csv"
audio_df.to_csv(save_path, index=False)
print(f"✅ Saved to: {save_path}")

✅ Saved to: /content/drive/MyDrive/urdu_toxic_audio_dataset.csv


In [ ]:
toxic_sample = audio_df[
    audio_df['new_BIO_tags'].apply(lambda x: 'B-Toxic' in x)
].iloc[0]

print("=== Toxic Sample Verification ===")
print(f"Original text:   {toxic_sample['original_text']}")
print(f"Transcription:   {toxic_sample['transcription']}")
print(f"Label:           {toxic_sample['label']}")
print(f"Original BIO:    {toxic_sample['original_BIO']}")
print(f"New BIO:         {toxic_sample['new_BIO_tags']}")
print(f"Toxic preserved: {toxic_sample['toxic_preserved']}")

ipd.display(ipd.Audio(toxic_sample['audio_path']))